In [17]:
import os
from pathlib import Path
import warnings

import pandas as pd
import numpy as np

import re

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize, sent_tokenize, TweetTokenizer, MWETokenizer
from nltk.probability import FreqDist

from gensim.models import Word2Vec

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt')
nltk.download('punkt_tab')

warnings.filterwarnings("ignore")

[nltk_data] Downloading package punkt to /Users/nicole/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nicole/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [18]:
linkedin = pd.read_csv('data/influencers_data.csv')
linkedin_data = linkedin.dropna(subset=['content'])['content']

In [31]:
tokenizer = TweetTokenizer()
initial_tokens = tokenizer.tokenize(linkedin_data[0])

li_tokens = linkedin_data.map(lambda x: tokenizer.tokenize(x))

li_tokens = li_tokens.reset_index(drop=True)
li_tokens = [[i for i in lst if any(c.isalnum() for c in i)] for lst in li_tokens]

In [40]:
def print_word_freq(tokens, target_words):
    if type(tokens[0]) == list:
        unpacked_tokens = [i for lst in tokens for i in lst]
        print(len(unpacked_tokens))
        fdist = FreqDist(unpacked_tokens)
    else:
        print(len(tokens))
        fdist = FreqDist(tokens)
    value_counts = {word : fdist[word] for word in target_words}
    print(value_counts)
    return value_counts

In [46]:
targets = ['build', 'ship', 'wonderful', 'business', 'fellow', 'friend']
li_target_freq = print_word_freq(li_tokens, targets)

1657038
{'build': 759, 'ship': 49, 'wonderful': 242, 'business': 1723, 'fellow': 101, 'friend': 354}


In [ ]:
def find_params(tokens):
    if type(tokens[0]) == list:
        unpacked_tokens = [i for lst in tokens for i in lst]
        tokens_count = len(unpacked_tokens)
    else:
        tokens_count = len(tokens)

    model_params = {
        'tiny' : {
            'vector_size' : 25,
            'min_count' : 3,
            'epochs' : 30,
            'negative' : 15
        },
        'small': {
            'vector_size' : 50,
            'min_count' : 10,
            'epochs' : 20,
            'negative' : 10
        },
        'medium' : {
            'vector_size' : 100,
            'min_count' : 10,
            'epochs' : 10,
            'negative' : 5
        },
        'large' : {
            'vector_size' : 100,
            'min_count' : 10,
            'epochs' : 5,
            'negative' : 5
        }
    }

    if tokens_count < 500000:
        model_size = 'tiny'
    elif tokens_count < 5000000:
        model_size = 'small'
    elif tokens_count < 10000000:
        model_size = 'medium'
    else:
        model_size = 'large'

    return model_params[model_size]

In [ ]:
li_model = Word2Vec(
    sentences=li_tokens, 
    vector_size=50,
    window=5,            
    min_count=10,     
    workers=4,
    sg=1,
    epochs=20,
    negative=10
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [157]:
li_model.wv.most_similar('build', topn=10)

[('create', 0.8357102274894714),
 ('develop', 0.7797974944114685),
 ('grow', 0.7566155195236206),
 ('drive', 0.7309370636940002),
 ('sustain', 0.719842791557312),
 ('bolster', 0.7130173444747925),
 ('acquire', 0.7085685133934021),
 ('generate', 0.6998188495635986),
 ('building', 0.6984800100326538),
 ('brand', 0.6925139427185059)]

In [158]:
li_model.wv.most_similar('leverage', topn=10)

[('optimize', 0.6935209631919861),
 ('strengths', 0.69189453125),
 ('extend', 0.66325843334198),
 ('grow', 0.6628800630569458),
 ('identify', 0.6603491306304932),
 ('hone', 0.6493343710899353),
 ('generate', 0.6463978886604309),
 ('Personal', 0.6333224177360535),
 ('maximize', 0.627371609210968),
 ('Align', 0.6234569549560547)]

In [159]:
li_model.wv.most_similar('wonderful')

[('lovely', 0.8555038571357727),
 ('amazing', 0.7847127914428711),
 ('great', 0.7656944990158081),
 ('beautiful', 0.7509565353393555),
 ('incredible', 0.7376995086669922),
 ('fun', 0.7249169945716858),
 ('inspiring', 0.7245922088623047),
 ('fantastic', 0.7155369520187378),
 ('tribute', 0.7068708539009094),
 ('Joan', 0.6957656145095825)]

In [160]:
li_model.wv.most_similar('create')

[('build', 0.8357102274894714),
 ('develop', 0.7736480832099915),
 ('shift', 0.727883517742157),
 ('drive', 0.7271252870559692),
 ('embrace', 0.7050800919532776),
 ('acquire', 0.7011961340904236),
 ('replicate', 0.700704038143158),
 ('creating', 0.6847632527351379),
 ('engage', 0.6845965385437012),
 ('adopt', 0.6822944283485413)]

In [161]:
li_model.wv.most_similar('business')

[('company', 0.7643189430236816),
 ('brand', 0.7366535663604736),
 ('strategy', 0.7280420064926147),
 ('firm', 0.7259805798530579),
 ('model', 0.7216526865959167),
 ('industry', 0.7169066667556763),
 ('product', 0.7118169069290161),
 ('building', 0.7100769877433777),
 ('success', 0.7036248445510864),
 ('culture', 0.697065532207489)]

In [35]:
def create_corpus(decade):
    dir_path = Path(f'data/coha-samples-text/{decade}')
    all_tokens = []

    for f in dir_path.iterdir():
        if f.is_file():
            try:
                with open(f, 'r', encoding='utf-8') as file:
                    content = file.readlines()
                    sentences = sent_tokenize(content[-1])
                    sentences = [word_tokenize(sent) for sent in sentences]
                    sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
                    all_tokens.extend(sentences)
            except Exception as e:
                print(f'Could not read {f.name}: {e}')
    
    return all_tokens

In [42]:
dummy_tokens = create_corpus('dummy')
pre1850_tokens = create_corpus('pre-1850')
pre1900_tokens = create_corpus('1850s-90s') + pre1850_tokens
d1900_tokens = create_corpus('1900s')
d1910_tokens = create_corpus('1910s')
d1920_tokens = create_corpus('1920s')
d1930_tokens = create_corpus('1930s')
d1940_tokens = create_corpus('1940s')
d1950_tokens = create_corpus('1950s')
d1960_tokens = create_corpus('1960s')
d1970_tokens = create_corpus('1970s')
d1980_tokens = create_corpus('1980s')
d1990_tokens = create_corpus('1990s')
d2000_tokens = create_corpus('2000s')

Could not read .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte
Could not read .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte


In [37]:
pre1900_tokens = create_corpus('1850s-90s') + pre1850_tokens
tokens_1900 = create_corpus('1900s')

In [ ]:
print_word_freq(d1900_tokens, targets)
print_word_freq(d1910_tokens, targets)
print_word_freq(d1920_tokens, targets)
print_word_freq(d1930_tokens, targets)
print_word_freq(d1940_tokens, targets)
print_word_freq(d1950_tokens, targets)
print_word_freq(d1960_tokens, targets)
print_word_freq(d1970_tokens, targets)
print_word_freq(d1980_tokens, targets)
print_word_freq(d1990_tokens, targets)
print_word_freq(d2000_tokens, targets)

234177
{'build': 23, 'ship': 46, 'wonderful': 19, 'business': 143, 'fellow': 99, 'friend': 97}
195147
{'build': 3, 'ship': 44, 'wonderful': 26, 'business': 54, 'fellow': 15, 'friend': 32}
223336
{'build': 14, 'ship': 31, 'wonderful': 21, 'business': 132, 'fellow': 16, 'friend': 28}
195847
{'build': 13, 'ship': 15, 'wonderful': 1, 'business': 91, 'fellow': 12, 'friend': 16}
322316
{'build': 10, 'ship': 160, 'wonderful': 9, 'business': 72, 'fellow': 22, 'friend': 52}
128604
{'build': 9, 'ship': 2, 'wonderful': 13, 'business': 52, 'fellow': 14, 'friend': 25}
250938
{'build': 19, 'ship': 42, 'wonderful': 10, 'business': 83, 'fellow': 17, 'friend': 30}
326300
{'build': 9, 'ship': 3, 'wonderful': 6, 'business': 44, 'fellow': 16, 'friend': 81}
199256
{'build': 13, 'ship': 28, 'wonderful': 5, 'business': 69, 'fellow': 12, 'friend': 34}
224495
{'build': 42, 'ship': 4, 'wonderful': 13, 'business': 51, 'fellow': 5, 'friend': 23}
247468
{'build': 18, 'ship': 24, 'wonderful': 12, 'business': 60, 'f

{'build': 18,
 'ship': 24,
 'wonderful': 12,
 'business': 60,
 'fellow': 15,
 'friend': 43}

In [ ]:
n1900_model = Word2Vec(
    sentences=tokens_1900, 
    vector_size=50,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    epochs=20,
    negative=10
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [163]:
n1900_model.wv.most_similar('build', topn=10)

[('trust', 0.6714775562286377),
 ('serve', 0.6697050333023071),
 ('provide', 0.6681744456291199),
 ('prevent', 0.6659179925918579),
 ('secure', 0.6518924236297607),
 ('assist', 0.6515113711357117),
 ('enter', 0.6464337706565857),
 ('develop', 0.6293811202049255),
 ('portions', 0.6191990375518799),
 ('1850', 0.6187852621078491)]

In [ ]:
pre1900_model = Word2Vec(
    sentences=pre1900_tokens, 
    vector_size=50,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    epochs=20,
    negative=10
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [165]:
pre1900_model.wv.most_similar('build', topn=10)

[('feed', 0.7074720859527588),
 ('adventurers', 0.6827496886253357),
 ('selecting', 0.6532171964645386),
 ('protect', 0.6415084600448608),
 ('yourselves', 0.6414114832878113),
 ('farms', 0.6333451867103577),
 ('hire', 0.6291592121124268),
 ('whites', 0.6203471422195435),
 ('furnaces', 0.619622528553009),
 ('strike', 0.6181607842445374)]

In [186]:
def create_coca_tokens(fp):
    dir_path = Path(fp)
    all_tokens = []

    for f in dir_path.iterdir():
        if f.is_file():
            try:
                with open(f, 'r', encoding='utf-8') as file:
                    content = file.readlines()
                    sentences = [re.sub(r'@@\d+', '', s) for s in content]
                    sentences = [sent_tokenize(s) for s in sentences]
                    sentences = [word_tokenize(s) for sent in sentences for s in sent]
                    sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
                    all_tokens.extend(sentences)
            except Exception as e:
                print(f'Could not read {f.name}: {e}')
    
    return all_tokens

In [188]:
coca_tokens = create_coca_tokens('data/coca-samples-text/')

In [189]:
sum([len(s) for s in coca_tokens])

9378210

In [ ]:
coca_model = Word2Vec(
    sentences=coca_tokens,
    vector_size=100,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    epochs=10,
    negative=5
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [193]:
coca_model.wv.most_similar('build', topn=10)

[('create', 0.7406520247459412),
 ('develop', 0.7308622598648071),
 ('innovate', 0.692916989326477),
 ('modernize', 0.6891975998878479),
 ('rebuild', 0.649296760559082),
 ('forge', 0.6476582288742065),
 ('allocate', 0.646988034248352),
 ('reconfigure', 0.6385540962219238),
 ('dismantle', 0.637188196182251),
 ('reconstruct', 0.6347952485084534)]

In [203]:
def create_enron_tokens(fp):
    dir_path = Path(fp)
    all_tokens = []

    for f in dir_path.iterdir():
        if f.is_file():
            try:
                with open(f, 'r', encoding='utf-8') as file:
                    content = file.readlines()
                    sentences = ''.join(content).replace('\n', ' ')
                    sentences = sent_tokenize(sentences)
                    sentences = [word_tokenize(s) for s in sentences]
                    sentences = [[token for token in tokens if any(c.isalnum() for c in token)] for tokens in sentences]
                    all_tokens.extend(sentences)
            except Exception as e:
                print(f'Could not read {f.name}: {e}')
    
    return all_tokens

In [205]:
enron_tokens = create_enron_tokens('data/enronsent')

Could not read .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte


In [208]:
sum([len(s) for s in enron_tokens])

13975435

In [ ]:
enron_model = Word2Vec(
    sentences=enron_tokens,
    vector_size=100,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    epochs=5,
    negative=5
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [212]:
enron_dist = FreqDist([t for line in enron_tokens for t in line])

In [214]:
enron_model.wv['build']

array([ 0.05960566, -0.13967216, -0.192964  , -0.34073707, -0.41870674,
       -0.23093434,  0.04985664,  0.49927938,  0.22975378, -0.41612378,
       -0.35994232, -0.6958026 ,  0.25747493,  0.01449603,  0.15946466,
       -0.1480282 , -0.29589376,  0.28780258, -0.18330908, -0.12624937,
       -0.10997437,  0.20908403,  0.0865448 , -0.577253  , -0.05241356,
       -0.05720013, -0.5927117 , -0.3148318 , -0.40854663,  0.00971841,
       -0.1797996 , -0.23060045,  0.20519073,  0.32311594, -0.22035463,
        0.09631245, -0.15192527, -0.14321151, -0.4104353 ,  0.1916394 ,
       -0.1985985 ,  0.13883348, -0.52788544, -0.5215233 ,  0.75597674,
        0.10496998, -0.48201478, -0.3537324 , -0.27254993,  0.13361508,
        0.24108045, -0.13600607, -0.07336839, -0.11009263,  0.15807141,
        0.3134331 ,  0.19610141,  0.03857709, -0.42260092,  0.38065958,
        0.66182375, -0.39915597,  0.11134535, -0.18690829, -0.3205693 ,
        0.03947898,  0.84460425,  0.3888131 , -0.17954965,  0.11

In [215]:
enron_model.wv.most_similar('build', topn=10)

[('develop', 0.7698425650596619),
 ('hinder', 0.702917754650116),
 ('spur', 0.6841839551925659),
 ('construct', 0.6794637441635132),
 ('maximise', 0.6786988973617554),
 ('modular', 0.6777182221412659),
 ('develope', 0.67405104637146),
 ('dynamically', 0.6730613112449646),
 ('forge', 0.6713972091674805),
 ('frustrate', 0.6711179614067078)]

In [216]:
enron_model.wv.most_similar('leverage', topn=10)

[('monetize', 0.7198493480682373),
 ('strengthen', 0.7019568085670471),
 ('optimize', 0.6980121731758118),
 ('productivity', 0.6946735978126526),
 ('complexities', 0.6944823861122131),
 ('teamwork', 0.6895453333854675),
 ('restructure', 0.6875494122505188),
 ('leveraging', 0.6845986843109131),
 ('navigate', 0.6827280521392822),
 ('disparate', 0.6818740963935852)]

In [217]:
coca_model.wv.most_similar('leverage', topn=10)

[('reshape', 0.7400580048561096),
 ('restructure', 0.7272158861160278),
 ('siphon', 0.7158498764038086),
 ('liquidity', 0.7049255967140198),
 ('extrapolate', 0.7009909152984619),
 ('garner', 0.6995500326156616),
 ('allocate', 0.697380781173706),
 ('stabilize', 0.6961700320243835),
 ('know-how', 0.6946137547492981),
 ('innovate', 0.6941611170768738)]